# Evaluation: Research Question Node
This notebook evaluates `research_question` predictions against benchmark labels and helps debug incorrect classifications.

In [8]:
# Setup imports and project paths
import os
import sys
import json
from pathlib import Path

import pandas as pd

# Resolve project root robustly from notebook location
ROOT = Path.cwd().resolve()
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent
if not (ROOT / "nodes").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PILOT = ROOT / "Pilot_Evaluation"
BENCHMARK_FILE = PILOT / "DATA_sample_10" / "Data Science Research Process (DSRP) Framework.csv"
RESULTS_FOLDER = PILOT / "pilot_study_results"

print(f"Project root: {ROOT}")
print(f"Benchmark file exists: {BENCHMARK_FILE.exists()}")
print(f"Results folder exists: {RESULTS_FOLDER.exists()}")

Project root: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow
Benchmark file exists: True
Results folder exists: True


In [9]:
# Load and clean benchmark labels
benchmark_df = pd.read_csv(BENCHMARK_FILE)
benchmark_df.columns = benchmark_df.columns.str.strip()

comment_cols = [
    c for c in benchmark_df.columns
    if "Any Comments" in c or "Any comments" in c
]
benchmark_df_clean = benchmark_df.drop(columns=comment_cols, errors="ignore")

required_cols = ["Paper ID", "Gatekeeper", "research_question"]
missing = [c for c in required_cols if c not in benchmark_df_clean.columns]
if missing:
    raise ValueError(f"Missing required benchmark columns: {missing}")

print(f"Loaded benchmark rows: {len(benchmark_df_clean)}")
print("Using columns:", required_cols)
print(benchmark_df_clean[required_cols].head())

Loaded benchmark rows: 10
Using columns: ['Paper ID', 'Gatekeeper', 'research_question']
    Paper ID Gatekeeper research_question
0   2011 - 1    Exclude               NaN
1   2018 - 8    Include        Predictive
2  2018 - 12    Exclude               NaN
3  2018 - 26    Exclude               NaN
4  2019 - 33    Include       Inferential


In [10]:
# Helper functions
def normalize_value(value):
    if pd.isna(value) or value is None:
        return None
    text = str(value).strip().lower()
    return text if text else None

def get_nested(data, path, default=None):
    current = data
    for key in path.split("."):
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current

def load_agent_results(results_folder: Path):
    agent_data = {}
    for paper_dir in sorted(results_folder.glob("*/")):
        if not paper_dir.is_dir():
            continue
        results_file = paper_dir / "aggregated" / "results.json"
        if not results_file.exists():
            continue
        with open(results_file, "r", encoding="utf-8") as f:
            agent_data[paper_dir.name.strip().lower()] = json.load(f)
    return agent_data

agent_data = load_agent_results(RESULTS_FOLDER)
print(f"Loaded aggregated agent outputs for {len(agent_data)} papers")

Loaded aggregated agent outputs for 10 papers


In [11]:
# Build comparison table for research_question (only Gatekeeper == include)
comparison_rows = []
included_rows = 0
skipped_not_included = 0

for _, row in benchmark_df_clean.iterrows():
    gatekeeper_label = normalize_value(row.get("Gatekeeper"))
    if gatekeeper_label != "include":
        skipped_not_included += 1
        continue

    included_rows += 1
    paper_id = normalize_value(row.get("Paper ID"))
    gt_label = normalize_value(row.get("research_question"))

    if not paper_id:
        continue
    if paper_id not in agent_data:
        print(f"Missing agent results for paper: {paper_id}")
        continue

    results = agent_data[paper_id]
    pred_label = get_nested(results, "dsrp_outputs.research_question.final_classification")

    # Fallbacks in case output schema changes
    if pred_label is None:
        pred_label = get_nested(results, "dsrp_outputs.research_question.classification")
    if pred_label is None:
        pred_label = get_nested(results, "dsrp_outputs.research_question.label")

    comparison_rows.append({
        "Paper ID": paper_id,
        "GT": gt_label,
        "Agent": normalize_value(pred_label)
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["Match"] = comparison_df.apply(
    lambda r: "✅" if r["GT"] == r["Agent"] else "❌", axis=1
)

print(f"Benchmark rows with Gatekeeper=include: {included_rows}")
print(f"Skipped (Gatekeeper!=include): {skipped_not_included}")
print(f"Comparison rows: {len(comparison_df)}")
print(comparison_df)

Benchmark rows with Gatekeeper=include: 7
Skipped (Gatekeeper!=include): 3
Comparison rows: 7
    Paper ID           GT        Agent Match
0   2018 - 8   predictive       causal     ❌
1  2019 - 33  inferential  inferential     ✅
2   2020 - 8  exploratory  exploratory     ✅
3  2021 - 28  inferential       causal     ❌
4  2021 - 56   predictive   predictive     ✅
5  2023 - 58  exploratory  exploratory     ✅
6  2024 - 99   predictive       causal     ❌


**Notes**
All 3 misclassified labels tagged *Causal* while two-times the ground truth (GT) was predictive, once GT was inferential. Therefore, I need to work on the rules for classifying causal's especially contrasting with *Predictive* and *Inferential* labels. 

**Before Prompt Fine-Tuning**

Benchmark rows with Gatekeeper=include: 7
Skipped (Gatekeeper!=include): 3
Comparison rows: 7
    Paper ID           GT        Agent Match
0   2018 - 8   predictive       causal     ❌
1  2019 - 33  inferential  inferential     ✅
2   2020 - 8  exploratory  exploratory     ✅
3  2021 - 28  inferential       causal     ❌
4  2021 - 56   predictive   predictive     ✅
5  2023 - 58  exploratory  exploratory     ✅
6  2024 - 99   predictive       causal     ❌

In [12]:
# Metrics for research_question
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

valid = comparison_df.dropna(subset=["GT", "Agent"]).copy()
y_true = valid["GT"].tolist()
y_pred = valid["Agent"].tolist()

if len(valid) == 0:
    raise ValueError("No valid GT/Agent pairs available for metrics")

metrics = {
    "valid_samples": len(valid),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
}

print("=" * 80)
print("RESEARCH QUESTION CLASSIFICATION METRICS")
print("=" * 80)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k:.<25} {v:.4f}")
    else:
        print(f"{k:.<25} {v}")

labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)
print("\nLabels:", labels)
print("Confusion Matrix:")
print(cm)

RESEARCH QUESTION CLASSIFICATION METRICS
valid_samples............ 7
accuracy................. 0.5714
precision_weighted....... 1.0000
recall_weighted.......... 0.5714
f1_weighted.............. 0.6905

Labels: ['causal', 'exploratory', 'inferential', 'predictive']
Confusion Matrix:
[[0 0 0 0]
 [0 2 0 0]
 [1 0 1 0]
 [2 0 0 1]]


In [13]:
# Detailed comparison and wrong-paper list
print("\nDETAILED COMPARISON: research_question\n")
print(comparison_df[["Paper ID", "GT", "Agent", "Match"]].to_string(index=False))

matches = (comparison_df["GT"] == comparison_df["Agent"]).sum()
total = len(comparison_df)
print(f"\nOverall Agreement: {matches}/{total} ({(100 * matches / total):.1f}%)")

wrong_paper_ids = comparison_df[comparison_df["Match"] == "❌"]["Paper ID"].tolist()
print(f"\nIncorrect classifications: {len(wrong_paper_ids)}")
if wrong_paper_ids:
    print(", ".join(wrong_paper_ids))


DETAILED COMPARISON: research_question

 Paper ID          GT       Agent Match
 2018 - 8  predictive      causal     ❌
2019 - 33 inferential inferential     ✅
 2020 - 8 exploratory exploratory     ✅
2021 - 28 inferential      causal     ❌
2021 - 56  predictive  predictive     ✅
2023 - 58 exploratory exploratory     ✅
2024 - 99  predictive      causal     ❌

Overall Agreement: 4/7 (57.1%)

Incorrect classifications: 3
2018 - 8, 2021 - 28, 2024 - 99


## Re-run Research Question Node for Incorrect Papers
After updating prompts in `prompts/dsrp/research_question/`, run the next cell to re-test only the incorrect papers.

In [33]:
from nodes.reasearch_question_node import research_question_node
from utils.dsrp_state import DSRPState

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in wrong_paper_ids:
    print(f"Re-running research_question node for {paper_id}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
    )

    try:
        out = research_question_node(state)
        pred = get_nested(out, "dsrp_outputs.research_question.final_classification")
        if pred is None:
            pred = get_nested(out, "dsrp_outputs.research_question.classification")
        if pred is None:
            pred = get_nested(out, "dsrp_outputs.research_question.label")

        gt = comparison_df.loc[comparison_df["Paper ID"] == paper_id, "GT"].iloc[0]
        retest_results.append({
            "Paper ID": paper_id,
            "GT": gt,
            "New Agent": normalize_value(pred),
        })
        print(f"  New prediction: {pred}")
    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "GT": comparison_df.loc[comparison_df["Paper ID"] == paper_id, "GT"].iloc[0],
            "New Agent": None,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    retest_df["Match"] = retest_df.apply(
        lambda r: "✅" if r.get("GT") == r.get("New Agent") else "❌", axis=1
    )
    print("\nRE-TEST RESULTS")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running research_question node for 2018 - 8...
  New prediction: Predictive
Re-running research_question node for 2021 - 28...
  New prediction: Inferential
Re-running research_question node for 2024 - 99...
  New prediction: Prescriptive

RE-TEST RESULTS
 Paper ID          GT    New Agent Match
 2018 - 8  predictive   predictive     ✅
2021 - 28 inferential  inferential     ✅
2024 - 99  predictive prescriptive     ❌


## Re-test on **correctly** classified labels earlier

In [30]:
# Detailed comparison and correct-paper list
print("\nDETAILED COMPARISON: research_question\n")
print(comparison_df[["Paper ID", "GT", "Agent", "Match"]].to_string(index=False))

matches = (comparison_df["GT"] == comparison_df["Agent"]).sum()
total = len(comparison_df)
print(f"\nOverall Agreement: {matches}/{total} ({(100 * matches / total):.1f}%)")

correct_paper_ids = comparison_df[comparison_df["Match"] != "❌"]["Paper ID"].tolist()
print(f"\nCorrect classifications: {len(correct_paper_ids)}")
if correct_paper_ids:
    print(", ".join(correct_paper_ids))


DETAILED COMPARISON: research_question

 Paper ID          GT       Agent Match
 2018 - 8  predictive      causal     ❌
2019 - 33 inferential inferential     ✅
 2020 - 8 exploratory exploratory     ✅
2021 - 28 inferential      causal     ❌
2021 - 56  predictive  predictive     ✅
2023 - 58 exploratory exploratory     ✅
2024 - 99  predictive      causal     ❌

Overall Agreement: 4/7 (57.1%)

Correct classifications: 4
2019 - 33, 2020 - 8, 2021 - 56, 2023 - 58


In [32]:
from nodes.reasearch_question_node import research_question_node
from utils.dsrp_state import DSRPState

# Ensure relative prompt paths in node code resolve correctly
original_dir = Path.cwd().resolve()
os.chdir(ROOT)

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

retest_results = []

for paper_id in correct_paper_ids:
    print(f"Re-running research_question node for {paper_id}...")
    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={},
        dsrp_outputs={},
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL,
    )

    try:
        out = research_question_node(state)
        pred = get_nested(out, "dsrp_outputs.research_question.final_classification")
        if pred is None:
            pred = get_nested(out, "dsrp_outputs.research_question.classification")
        if pred is None:
            pred = get_nested(out, "dsrp_outputs.research_question.label")

        gt = comparison_df.loc[comparison_df["Paper ID"] == paper_id, "GT"].iloc[0]
        retest_results.append({
            "Paper ID": paper_id,
            "GT": gt,
            "New Agent": normalize_value(pred),
        })
        print(f"  New prediction: {pred}")
    except Exception as e:
        retest_results.append({
            "Paper ID": paper_id,
            "GT": comparison_df.loc[comparison_df["Paper ID"] == paper_id, "GT"].iloc[0],
            "New Agent": None,
            "Error": str(e),
        })
        print(f"  Error: {e}")

os.chdir(original_dir)

retest_df = pd.DataFrame(retest_results)
if len(retest_df):
    retest_df["Match"] = retest_df.apply(
        lambda r: "✅" if r.get("GT") == r.get("New Agent") else "❌", axis=1
    )
    print("\nRE-TEST RESULTS")
    print(retest_df.to_string(index=False))
else:
    print("No incorrect papers to re-test.")

Re-running research_question node for 2019 - 33...
  New prediction: Inferential
Re-running research_question node for 2020 - 8...
  New prediction: Exploratory
Re-running research_question node for 2021 - 56...
  New prediction: Predictive
Re-running research_question node for 2023 - 58...
  New prediction: Exploratory

RE-TEST RESULTS
 Paper ID          GT   New Agent Match
2019 - 33 inferential inferential     ✅
 2020 - 8 exploratory exploratory     ✅
2021 - 56  predictive  predictive     ✅
2023 - 58 exploratory exploratory     ✅
